In [1]:
from pathlib import Path

import pandas as pd

BLUE, GREEN, YELLOW, RED, BOLD, RESET = "\033[94m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"


def step(msg):
    print(f"\n{BOLD}{BLUE}▶ {msg}{RESET}")


def done(msg):
    print(f"{GREEN}  ✓ {msg}{RESET}")


def note(msg):
    print(f"{YELLOW}  • {msg}{RESET}")


ROOT = Path.cwd()
while not (ROOT / "data" / "master_listing.parquet").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MASTER = ROOT / "data" / "master_listing.parquet"
FUNNEL = ROOT / "data" / "feature_funnel.csv"
CATALOG = ROOT / "data" / "master_columns.csv"
OUT = ROOT / "data" / "feature_funnel_numerical.csv"

TYPES = ["number"]                        # add "boolean" to also load the 223 flags
GROUP_KEYS = ["unit_id", "property_id"]   # property_id is needed for GroupKFold

step(f"Reading {FUNNEL.name}")
funnel = pd.read_csv(FUNNEL)
done(f"{len(funnel)} variables in the funnel")

step(f"Keeping type in {TYPES}")
num = funnel[funnel["type"].isin(TYPES)].copy()
done(f"{len(num)} numerical variables ({len(funnel) - len(num)} dropped)")

step(f"Tagging each variable with its role from {CATALOG.name}")
# The role says whether a column may go in X: only `feature` (and `inferred`) may.
num = num.merge(pd.read_csv(CATALOG)[["column", "role"]], left_on="variable", right_on="column").drop(columns="column")
for role, n in num["role"].value_counts().items():
    color = GREEN if role in ("feature", "inferred") else YELLOW if role == "label" else RED
    usage = {"feature": "usable as X", "inferred": "usable as X", "label": "targets",
             "leaky": "NEVER in X: this listing's own price", "latent": "NEVER in X: not observable"}[role]
    print(f"{color}  • {role:9s}{n:>4}  {usage}{RESET}")

step(f"Loading {len(num)} numerical columns from {MASTER.name}")
df_num = pd.read_parquet(MASTER, columns=GROUP_KEYS + num["variable"].tolist())
done(f"{df_num.shape[0]:,} rows x {df_num.shape[1]} columns "
     f"({df_num.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory)")

X_cols = num.loc[num["role"].isin(["feature", "inferred"]), "variable"].tolist()
label_cols = num.loc[num["role"] == "label", "variable"].tolist()
note(f"X_cols:     {len(X_cols)} candidate features")
note(f"label_cols: {len(label_cols)} possible targets")

step(f"Writing {OUT.relative_to(ROOT)}")
num.to_csv(OUT, index=False)
done(f"{len(num)} rows x {num.shape[1]} columns")
df_num.head()


▶ Reading feature_funnel.csv
  ✓ 378 variables in the funnel

▶ Keeping type in ['number']
  ✓ 108 numerical variables (270 dropped)

▶ Tagging each variable with its role from master_columns.csv
  • feature    73  usable as X
  • leaky      14  NEVER in X: this listing's own price
  • label      11  targets
  • latent      9  NEVER in X: not observable
  • inferred    1  usable as X

▶ Loading 108 numerical columns from master_listing.parquet


  ✓ 967,661 rows x 110 columns (902 MB in memory)
  • X_cols:     74 candidate features
  • label_cols: 11 possible targets

▶ Writing data/feature_funnel_numerical.csv
  ✓ 108 rows x 6 columns


,unit_id,property_id,market_unit__bed,market_unit__bath,market_unit__partial_bath,market_unit__floor,market_unit__sqft,market_unit__days_on_market,truth_lease_event__true_term_months,truth_lease_event__true_rent,...,truth_concession__max_total_value,truth_concession__max_amortized_monthly,truth_concession__n_generic,truth_concession__n_net_effective,market_leasing_activity_daily__leased_pct_at_enter,market_leasing_activity_daily__exposure_pct_at_enter,truth_floorplan_market_rent__true_market_rent_at_enter,truth_floorplan_market_rent__true_market_rent_psf_at_enter,pms_property__num_available_vacant_units,pms_property__num_available_notice_units
0,9f87b068-L000226,9f87b068-f782-40f3-855e-0e7e587eb4df,1,1,0,5.0,850.0,31,12.0,1240.0,...,1255.0,83.67,0.0,1.0,0.9759,0.0602,1287.27,1.51,7.0,12.0
1,9f87b068-L000349,9f87b068-f782-40f3-855e-0e7e587eb4df,1,1,0,18.0,850.0,20,6.0,1340.0,...,1209.0,100.75,0.0,0.0,1.0000,0.0181,1339.45,1.58,7.0,12.0
2,a30f7758-L000163,a30f7758-ddd1-4253-97c0-092044f69c99,2,2,0,NaN,905.0,89,7.0,1350.0,...,1345.0,112.08,0.0,0.0,1.0000,0.0577,1405.98,1.55,7.0,8.0
3,a4d71e3f-L000543,a4d71e3f-01fc-4706-a50d-48b5a83a70d4,0,1,0,1.0,540.0,26,7.0,1565.0,...,1500.0,100.00,0.0,0.0,0.9833,0.0628,1636.41,3.03,6.0,13.0
4,7d7f3f38-L000484,7d7f3f38-52cb-4537-b9ed-808178cf9d8a,1,1,0,NaN,785.0,115,12.0,1800.0,...,600.0,40.00,0.0,0.0,0.9226,0.0903,1776.33,2.26,19.0,26.0


## Feature selection

Scores every candidate feature in `X_cols` against the target three ways and writes
`data/feature_selection_numerical.csv` (one row per feature, strongest first by gradient boosting):

| column | method | how to read it |
| --- | --- | --- |
| `gini` | Credit-scoring Gini for a continuous target (normalized Gini of the Lorenz curve), on every listing where the feature is present | 1 = ranks rent perfectly, 0 = no better than random, negative = higher value means lower rent. Rank by absolute value. For a 0/1 target it equals 2·AUC − 1. |
| `gradient_boosting` | R² lost on unseen buildings when the feature is shuffled (permutation importance of a gradient-boosted model) | about 0 = the model does not need it. Counts only what the feature adds beyond the others. |
| `lasso` | Coefficient of an L1-penalized linear regression on standardized features | change in rent ($) per one standard deviation of the feature. Exactly 0 = Lasso dropped it. |

Gini looks at one feature at a time; gradient boosting and Lasso look at all of them together,
so a feature that repeats another one can have a high Gini and a low score on the other two.

- A low-completeness feature's Gini only covers the rows it is present on: read it next to `completeness_pct`.
- Both models are validated with `GroupKFold` by `property_id`, so they are scored on buildings they never saw.
- Some features are only known after the listing closes (`market_unit__days_on_market`, the price-change
  and concession counts). Decide when the prediction is made before keeping them.
- `GINI_MIN` and `GB_MIN` only drive the printed summary; the CSV keeps the raw scores.

In [2]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

TARGET = "truth_lease_event__true_rent"   # or truth_floorplan_market_rent__true_market_rent_psf_at_enter
SAMPLE = 300_000                          # listings used by the two models; Gini uses all of them
N_FOLDS = 3
SEED = 42
GINI_MIN, GB_MIN = 0.10, 0.001            # summary thresholds only
SELECTION_OUT = ROOT / "data" / "feature_selection_numerical.csv"


def gini(x, y):
    """Normalized Gini of ordering y by x.

    Sort by x from high to low, take the Gini of the Lorenz curve of cumulative y, and divide by
    the Gini of the perfect ordering (y by itself). Tied x values are taken as one block, so a
    feature with few distinct values (bed, bath) is not scored on an arbitrary tie order.
    """
    def lorenz_gini(key):
        g = pd.DataFrame({"k": key, "y": y}).groupby("k")["y"].agg(["size", "sum"]).iloc[::-1]
        px = np.r_[0, g["size"].cumsum() / len(y)]
        py = np.r_[0, g["sum"].cumsum() / y.sum()]
        return 2 * ((py[1:] + py[:-1]) / 2 * np.diff(px)).sum() - 1
    return lorenz_gini(x) / lorenz_gini(y)


step(f"Preparing the target: {TARGET}")
data = df_num[df_num[TARGET].notna()]
scored = num.loc[num["variable"].isin(X_cols) & (num["completeness_pct"] > 0), "variable"].tolist()
done(f"{len(data):,} listings with a known target ({len(df_num) - len(data):,} without one dropped)")
note(f"{len(scored)} features to score, {len(X_cols) - len(scored)} skipped because they are 100% empty")

step("Gini of each feature vs the target")
scores = pd.DataFrame(index=scored)
gini_scores = []
for c in scored:
    present = data[c].notna()
    gini_scores.append(gini(data.loc[present, c].to_numpy(), data.loc[present, TARGET].to_numpy()))
scores["gini"] = gini_scores
done(f"|Gini| >= {GINI_MIN} for {(scores['gini'].abs() >= GINI_MIN).sum()} of {len(scored)} features")


▶ Preparing the target: truth_lease_event__true_rent


  ✓ 931,738 listings with a known target (35,923 without one dropped)
  • 61 features to score, 13 skipped because they are 100% empty

▶ Gini of each feature vs the target


  ✓ |Gini| >= 0.1 for 38 of 61 features


In [3]:
step(f"Gradient boosting: {N_FOLDS}-fold GroupKFold by building on {SAMPLE:,} sampled listings (~3 min)")
# Grouped by property_id so no building is in both train and validation: otherwise the
# building columns would score well just by letting the model memorize buildings.
sample = data.sample(min(SAMPLE, len(data)), random_state=SEED)
X = sample[scored].astype("float64")
y = sample[TARGET].to_numpy()
folds = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups=sample["property_id"]))

importances = []
for i, (train, valid) in enumerate(folds, 1):
    model = HistGradientBoostingRegressor(max_iter=300, random_state=SEED).fit(X.iloc[train], y[train])
    note(f"fold {i}/{N_FOLDS}: R² on unseen buildings = {model.score(X.iloc[valid], y[valid]):.3f}, shuffling features ...")
    perm = permutation_importance(model, X.iloc[valid], y[valid], scoring="r2",
                                  n_repeats=3, random_state=SEED, n_jobs=-1)
    importances.append(perm.importances_mean)
scores["gradient_boosting"] = np.mean(importances, axis=0)
done(f"importance > {GB_MIN} for {(scores['gradient_boosting'] > GB_MIN).sum()} of {len(scored)} features")


▶ Gradient boosting: 3-fold GroupKFold by building on 300,000 sampled listings (~3 min)


  • fold 1/3: R² on unseen buildings = 0.784, shuffling features ...


  • fold 2/3: R² on unseen buildings = 0.790, shuffling features ...


  • fold 3/3: R² on unseen buildings = 0.789, shuffling features ...


  ✓ importance > 0.001 for 29 of 61 features


In [4]:
step("Lasso: L1-penalized linear regression on standardized features")
# Median-imputed and standardized, so each coefficient is the change in rent ($) for one
# standard deviation of the feature. Alpha is the largest one whose CV error is within one
# standard error of the best (the "1-SE rule"): the sparsest model that is about as good.
Xs = StandardScaler().fit_transform(X.fillna(X.median()))
cv = LassoCV(alphas=50, cv=folds, max_iter=5000, n_jobs=-1).fit(Xs, y)
mse = cv.mse_path_.mean(axis=1)
se = cv.mse_path_.std(axis=1) / np.sqrt(N_FOLDS)
alpha = cv.alphas_[mse <= mse.min() + se[mse.argmin()]].max()
scores["lasso"] = Lasso(alpha=alpha, max_iter=5000).fit(Xs, y).coef_
done(f"alpha = {alpha:.2f}: Lasso kept {(scores['lasso'] != 0).sum()} of {len(scored)} features, the rest are exactly 0")


▶ Lasso: L1-penalized linear regression on standardized features


  ✓ alpha = 15.02: Lasso kept 24 of 61 features, the rest are exactly 0


In [5]:
step(f"Writing {SELECTION_OUT.relative_to(ROOT)}")
selection = (num.loc[num["variable"].isin(X_cols), ["source_table", "variable", "completeness_pct"]]
             .rename(columns={"variable": "feature"})
             # + 0.0 turns Lasso's -0.0 into 0.0
             .join(scores.round({"gini": 4, "gradient_boosting": 4, "lasso": 2}) + 0.0, on="feature")
             .sort_values("gradient_boosting", ascending=False, na_position="last")
             .reset_index(drop=True))
selection.to_csv(SELECTION_OUT, index=False)
done(f"{len(selection)} features x {selection.shape[1]} columns")

kept = {
    f"|gini| >= {GINI_MIN}": selection["gini"].abs() >= GINI_MIN,
    f"gradient boosting > {GB_MIN}": selection["gradient_boosting"] > GB_MIN,
    "lasso != 0": selection["lasso"].fillna(0) != 0,
}
all_three = np.logical_and.reduce(list(kept.values()))

step("Funnel")
for label, n in [("candidate features", len(selection)), ("non-empty", len(scored))] + [
        (label, mask.sum()) for label, mask in kept.items()] + [("kept by all three", all_three.sum())]:
    color = GREEN if n / len(selection) >= 0.5 else YELLOW if n / len(selection) >= 0.25 else RED
    print(f"  {label:28s}{color}{'█' * round(40 * n / len(selection)):40s} {n:>3}{RESET}")

step("Kept by all three methods")
for r in selection[all_three].itertuples():
    print(f"{GREEN}  ✓ {r.feature:52s}{RESET} gini {r.gini:+.3f}   gb {r.gradient_boosting:.4f}   lasso {r.lasso:+9.2f}")
selection.head(25)


▶ Writing data/feature_selection_numerical.csv
  ✓ 74 features x 6 columns

▶ Funnel
  candidate features          ████████████████████████████████████████  74
  non-empty                   █████████████████████████████████         61
  |gini| >= 0.1               █████████████████████                     38
  gradient boosting > 0.001   ████████████████                          29
  lasso != 0                  █████████████                             24
  kept by all three           ██████████                                18

▶ Kept by all three methods
  ✓ market_unit__sqft                                    gini +0.540   gb 0.3760   lasso   +302.17
  ✓ market_building__n_building_amenities                gini +0.507   gb 0.1794   lasso   +158.71
  ✓ market_building__lon                                 gini +0.399   gb 0.1493   lasso   +274.47
  ✓ market_building__n_unit_amenities                    gini +0.500   gb 0.0118   lasso    +35.13
  ✓ market_building__avg_price_change  

,source_table,feature,completeness_pct,gini,gradient_boosting,lasso
0,market_unit,market_unit__sqft,95.67,0.5399,0.3760,302.17
1,market_building,market_building__lat,100.00,0.1937,0.2798,0.00
2,market_building,market_building__n_building_amenities,82.79,0.5073,0.1794,158.71
3,market_building,market_building__lon,100.00,0.3987,0.1493,274.47
4,market_building,market_building__year_built,89.22,0.0903,0.0535,69.27
5,market_unit,market_unit__bed,100.00,0.3738,0.0414,0.00
6,market_building,market_building__n_unit_amenities,82.79,0.4999,0.0118,35.13
7,market_building,market_building__avg_time_on_market_days,99.99,0.0623,0.0115,24.63
8,market_building,market_building__avg_price_change,100.00,-0.1023,0.0095,-1.68
9,market_building,market_building__avg_review_score,100.00,0.3001,0.0077,38.52
